# D1-SOURCE-FAITHFULNESS-v2 — four-arm local evaluation

**Study question:** Does a contrastively fitted LiquidAI/d1-3B activation direction improve detection of unsupported source claims on RAGTruth, beyond the unsteered model, instruction steering, and a random-direction sham?

This is **not** a replacement for FEMI's deterministic trusted-record authorization gate. No OpenRouter calls. Select a **GPU** runtime before starting (L4/A100 recommended if available). An unsuccessful or incompatible activation hook is a legitimate stopped run, **not** evidence that steering worked.

Read `PROTOCOL.md` before the heldout cell; the freeze is local, not an external preregistration.

## 0 — Upload this experiment ZIP and enter its root

In [ ]:
from google.colab import files
from pathlib import Path
import os, zipfile, subprocess, json, shutil, hashlib

root = Path('/content/D1_SOURCE_FAITHFULNESS_V2_20261108')
if not (root/'src/d1_experiment.py').exists():
    uploaded = files.upload()  # select D1_SOURCE_FAITHFULNESS_V2_20261108.zip
    packet = next((Path('/content')/name for name in uploaded if name.startswith('D1_SOURCE_FAITHFULNESS_V2_') and name.endswith('.zip')), None)
    assert packet and packet.exists(), 'Upload D1_SOURCE_FAITHFULNESS_V2_20261108.zip'
    with zipfile.ZipFile(packet) as z:
        z.extractall('/content')
assert (root/'src/d1_experiment.py').is_file()
os.chdir(root)
print('Working directory:',os.getcwd())

## 1 — Offline software tests (synthetic fixtures only)

In [ ]:
subprocess.run(['python','-m','unittest','discover','-s','tests','-v'],check=True)
print('These tests do not run Liquid d1 or evaluate public benchmark examples.')

## 2 — Acquire RAGTruth and record provenance

Source: [ParticleMedia/RAGTruth](https://github.com/ParticleMedia/RAGTruth). The repository has `dataset/response.jsonl` and `dataset/source_info.jsonl`. Use the official repository and record the commit, rather than copying a third-party converted dataset.

In [ ]:
repo = Path('/content/RAGTruth')
if not (repo/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','https://github.com/ParticleMedia/RAGTruth.git',str(repo)],check=True)
commit=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
source_file=repo/'dataset/source_info.jsonl'
response_file=repo/'dataset/response.jsonl'
assert source_file.is_file() and response_file.is_file(), 'Official RAGTruth data files missing'
print('Dataset repository commit:',commit)
print('Input files:',source_file.stat().st_size,response_file.stat().st_size)
(root/'RAGTRUTH_GIT_COMMIT.txt').write_text(commit+'\n')

## 3 — Freeze the dataset **once**, inspect counts and source separation

Train: 24 matched opposite-label source pairs; dev: 60 distinct sources; test: 120 distinct sources, balanced within QA/Summary/Data2txt and labels. The original public RAGTruth **test split** is held out. The model input never contains gold labels.

In [ ]:
run_dir=Path('runs/d1')
if not (run_dir/'FREEZE.json').exists():
    subprocess.run(['python','src/prepare.py',
        '--source-file',str(source_file),'--response-file',str(response_file),
        '--out',str(run_dir)],check=True)
subprocess.run(['python','src/d1_experiment.py','preflight','--dir',str(run_dir)],check=True)
freeze=json.loads((run_dir/'FREEZE.json').read_text())
print('Source-disjoint, label-balanced allocation:')
print(json.dumps(freeze['per_partition_task_label'],indent=2))
assert freeze['n_train']==48 and freeze['n_dev']==60 and freeze['n_test']==120
print('Dataset SHA-256:',freeze['data_sha256'])

## 4 — Install d1 dependencies and pin checkpoint before touching test

`LiquidAI/d1-3B` publishes `system_one` for decision probabilities. Running it here requires publisher-supplied remote model code (`trust_remote_code=True`), so review the model card and license. Keep the exact revision fixed across probe/fit/test. The 3B model may not fit every free GPU.

In [ ]:
subprocess.run(['pip','install','-q','-r','requirements.txt'],check=True)
import torch
assert torch.cuda.is_available(), 'Switch Colab Runtime to GPU, then rerun this cell'
print('GPU:',torch.cuda.get_device_name(0))
from huggingface_hub import model_info
pinfile=root/'D1_HF_REVISION.txt'
if pinfile.exists():
    revision=pinfile.read_text().strip()
else:
    revision=model_info('LiquidAI/d1-3B').sha
    assert len(revision)>=12
    pinfile.write_text(revision+'\n')
os.environ['D1_HF_REVISION']=revision
print('Frozen d1 checkpoint:',revision)

## 5 — TRAIN-only activation hook preflight

Automatically identify a full transformer block that fires exactly once and returns `[batch,sequence,hidden]`. Selection uses module structure and one training example, **not dev/test labels**. A failure means the current publisher implementation is not safely steerable with this hook; stop rather than guessing a layer.

In [ ]:
subprocess.run(['python','src/d1_experiment.py','probe','--dir',str(run_dir)],check=True)
info=json.loads((run_dir/'hook_probe.json').read_text())
print('Selected module:',info['hook_name'])
print('Observed hidden width:',info['shape'][-1])

## 6 — Fit on TRAIN; select dose on DEV only

Direction uses matched source pairs, not synthetic token substitutions. DEV selects among fixed nonzero doses with a supported-acceptance guardrail. If no dose improves, the run continues with a labeled, predetermined fallback; this must remain a negative result if the heldout outcome fails.

In [ ]:
if not (run_dir/'d1_fitted.pt').exists():
    subprocess.run(['python','src/d1_experiment.py','fit','--dir',str(run_dir)],check=True)
else:
    print('Existing fitted direction found. Preserve it; do not refit after inspecting heldout results.')

## 7 — HELD-OUT FOUR-ARM TEST (120 × 4 = 480 local d1 decisions)

**Run once only.** This is the locked test. An interrupted run writes a validated `d1_heldout_PARTIAL.jsonl`; rerunning this cell resumes. Completed records cannot be overwritten. No outcome-driven prompt/layer/dose changes are allowed.

In [ ]:
if not (run_dir/'d1_heldout_predictions.jsonl').exists():
    subprocess.run(['python','src/d1_experiment.py','test','--dir',str(run_dir)],check=True)
else:
    print('Heldout run already completed. No repeated inference.')
subprocess.run(['python','src/d1_experiment.py','analyze','--dir',str(run_dir)],check=True)

## 8 — Optional source-withholding diagnostic (after test only)

Same candidate, supplied source removed, **60 positive original test items × 2 arms**. Only evaluate source-sensitivity; these are not newly human-labeled counterfactual examples.

In [ ]:
if not (run_dir/'d1_source_withheld.jsonl').exists():
    subprocess.run(['python','src/d1_experiment.py','ablate','--dir',str(run_dir)],check=True)
else:
    print('Source-removal results already saved. No repeated inference.')

## 9 — Export complete research archive for independent audit

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile,hashlib
archive=Path('/content/D1_SOURCE_FAITHFULNESS_V2_RESULTS.zip')
include=[root/'PROTOCOL.md',root/'README.md',root/'RAGTRUTH_GIT_COMMIT.txt',root/'D1_HF_REVISION.txt',
         root/'src/prepare.py',root/'src/d1_experiment.py',root/'requirements.txt']
include+=list(run_dir.glob('*'))
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in include:
        if p.is_file():z.write(p,arcname=str(p.relative_to(root)))
print('Archive created:',archive,'size MB:',round(archive.stat().st_size/1e6,2))
print('SHA-256:',hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))